# Build Knowledge Edges (2022–2025 Data)

**Purpose:** Process all 2022-2025 triplet weight CSV files to build knowledge edges in the same format as the existing `knowledge_edges.csv`.

**Key differences from old `09_knowledge_edges.ipynb`:**
- `paper_ID` in new triplets is already an OpenAlex W-format ID (e.g. `W4319662928`)
- We construct `global_id = {SPLIT}_{DOMAIN}_{W-ID}` directly — no local integer mapping
- Using `*_weights.csv` files (have valid pred_weights) but dropping pred_weights column for consistency
- New entities not in existing entity_nodes are added to entity_nodes

**Output:** `knowledge_edges_2022_2025.csv`, updated `entity_nodes.csv`

In [1]:
import pandas as pd
import hashlib
import os
from glob import glob

# Paths
TRIPLETS_DIR = "../Scientific_Novelty_Detection_2022_2025/data(2022-2025)/Triplets/"
OUT_DIR      = "../outputs/final/"

In [2]:
TOPIC_TO_DOMAIN = {
    # Novel_Papers files
    'dia2022_2025'  : 'DIA',
    'mt2022_2025'   : 'MT',
    'qa2022_2025'   : 'QA',
    'sa2022_2025'   : 'SA',
    'sum2022_2025'  : 'SUM',
    # Blogs + SKG files (topic field is just the bare domain name)
    'dia'           : 'DIA',
    'mt'            : 'MT',
    'nli'           : 'NLI',
    'par'           : 'PAR',
    'qa'            : 'QA',
    'sa'            : 'SA',
    'sum'           : 'SUM',
}

FOLDER_TO_SPLIT = {
    'NOVEL_PAPERS' : 'NOVEL',
    'BLOGS'        : 'BLOG',
    'SKG'          : 'SKG',
}

print("Config loaded.")

Config loaded.


In [3]:
# Load existing entity_nodes and paper_nodes
entity_nodes = pd.read_csv(OUT_DIR + 'entity_nodes.csv')
paper_nodes  = pd.read_csv(OUT_DIR + 'paper_nodes.csv')
new_pn       = pd.read_csv(OUT_DIR + 'paper_nodes_2022_2025.csv')

# Build entity map: name → node_id
entity_map = dict(zip(entity_nodes['name'], entity_nodes['node_id']))

# Build paper ID set from BOTH old and new paper nodes
all_paper_ids = set(paper_nodes['node_id']) | set(new_pn['node_id'])

# Build openalex_id → node_id lookup for new papers
new_id_lookup = dict(zip(new_pn['openalex_id'], new_pn['node_id']))
# Also year lookup
new_year_lookup = dict(zip(new_pn['node_id'], new_pn['year']))

print(f"Existing entity nodes: {len(entity_nodes)}")
print(f"Old paper nodes:       {len(paper_nodes)}")
print(f"New paper nodes:       {len(new_pn)}")
print(f"Combined paper IDs:    {len(all_paper_ids)}")

Existing entity nodes: 141728
Old paper nodes:       2529
New paper nodes:       2331
Combined paper IDs:    4860


In [4]:
# Find all weight CSV files
weight_files = glob(os.path.join(TRIPLETS_DIR, '**', '*_weights.csv'), recursive=True)
print(f"Weight files found: {len(weight_files)}")
for f in sorted(weight_files):
    print(" ", f.replace(TRIPLETS_DIR, ''))

Weight files found: 19
  ../Scientific_Novelty_Detection_2022_2025/data(2022-2025)/Triplets\Blogs\Dia_triplets_weights.csv
  ../Scientific_Novelty_Detection_2022_2025/data(2022-2025)/Triplets\Blogs\MT_triplets_weights.csv
  ../Scientific_Novelty_Detection_2022_2025/data(2022-2025)/Triplets\Blogs\NLI_triplets_weights.csv
  ../Scientific_Novelty_Detection_2022_2025/data(2022-2025)/Triplets\Blogs\Par_triplets_weights.csv
  ../Scientific_Novelty_Detection_2022_2025/data(2022-2025)/Triplets\Blogs\QA_triplets_weights.csv
  ../Scientific_Novelty_Detection_2022_2025/data(2022-2025)/Triplets\Blogs\SA_triplets_weights.csv
  ../Scientific_Novelty_Detection_2022_2025/data(2022-2025)/Triplets\Blogs\Sum_triplets_weights.csv
  ../Scientific_Novelty_Detection_2022_2025/data(2022-2025)/Triplets\Novel_Papers\Dia2022_2025_triplets_weights.csv
  ../Scientific_Novelty_Detection_2022_2025/data(2022-2025)/Triplets\Novel_Papers\MT2022_2025_triplets_weights.csv
  ../Scientific_Novelty_Detection_2022_2025/data(

In [5]:
def normalize_entity(text):
    if pd.isna(text):
        return None
    text = str(text).strip().lower()
    text = text.replace('"""', '"')
    text = text.replace("''", "'")
    text = ' '.join(text.split())
    return text


def generate_entity_id(entity_string):
    return 'E_' + hashlib.md5(entity_string.encode('utf-8')).hexdigest()[:10]

print("Functions defined.")

Functions defined.


In [6]:
edges        = []
new_entities = {}   # name → node_id for entities not yet in entity_nodes
skipped      = set()

for filepath in sorted(weight_files):

    # Determine split from parent folder name
    folder_name = os.path.basename(os.path.dirname(filepath)).upper()
    split = FOLDER_TO_SPLIT.get(folder_name)
    if split is None:
        print(f"  ⚠ Unknown folder '{folder_name}' — skipping")
        continue

    df = pd.read_csv(filepath)
    required = ['sub', 'obj', 'pred', 'paper_ID', 'topic']
    if not all(c in df.columns for c in required):
        print(f"  ⚠ Missing columns in {os.path.basename(filepath)}")
        continue

    file_edges = 0

    for _, row in df.iterrows():
        paper_id_raw = str(row['paper_ID']).strip()

        # Get topic → domain
        topic  = str(row['topic']).strip().lower().replace(' ', '_')
        domain = TOPIC_TO_DOMAIN.get(topic)
        if domain is None:
            # Try stripping trailing numbers
            domain = TOPIC_TO_DOMAIN.get('_'.join(topic.split('_')[:-1]))
        if domain is None:
            skipped.add(topic)
            continue

        # Build global_id
        global_id = f"{split}_{domain}_{paper_id_raw}"

        if global_id not in all_paper_ids:
            skipped.add(global_id)
            continue

        # Get year
        source_year = new_year_lookup.get(global_id)
        if source_year is None:
            continue

        sub       = normalize_entity(row['sub'])
        obj       = normalize_entity(row['obj'])
        predicate = row['pred']
        # pred_weights dropped for consistency with old pipeline

        for entity in [sub, obj]:
            if entity is None:
                continue

            # Use existing entity or create new one
            if entity in entity_map:
                entity_id = entity_map[entity]
            elif entity in new_entities:
                entity_id = new_entities[entity]
            else:
                entity_id = generate_entity_id(entity)
                new_entities[entity] = entity_id
                entity_map[entity]   = entity_id  # add to lookup immediately

            edges.append({
                'source'    : global_id,
                'target'    : entity_id,
                'predicate' : predicate,
                'year'      : int(source_year),
            })
            file_edges += 1

    print(f"  ✓ {os.path.basename(filepath):50s} split={split:6s} edges={file_edges}")

print(f"\nTotal raw edges: {len(edges)}")
print(f"New entities discovered: {len(new_entities)}")
if skipped:
    print(f"Skipped topics/IDs (sample): {list(skipped)[:10]}")

  ✓ Dia_triplets_weights.csv                           split=BLOG   edges=40603
  ✓ MT_triplets_weights.csv                            split=BLOG   edges=29103
  ✓ NLI_triplets_weights.csv                           split=BLOG   edges=33069
  ✓ Par_triplets_weights.csv                           split=BLOG   edges=42731
  ✓ QA_triplets_weights.csv                            split=BLOG   edges=35728
  ✓ SA_triplets_weights.csv                            split=BLOG   edges=37531
  ✓ Sum_triplets_weights.csv                           split=BLOG   edges=37112
  ✓ Dia2022_2025_triplets_weights.csv                  split=NOVEL  edges=35570
  ✓ MT2022_2025_triplets_weights.csv                   split=NOVEL  edges=15671
  ✓ QA2022_2025_triplets_weights.csv                   split=NOVEL  edges=2942
  ✓ SA2022_2025_triplets_weights.csv                   split=NOVEL  edges=12902
  ✓ Sum2022_2025_triplets_weights.csv                  split=NOVEL  edges=1186
  ✓ Dia_triplets_weights.csv              

In [7]:
# Build edges DataFrame and deduplicate
new_edges = pd.DataFrame(edges)
print(f"Raw edges:        {len(new_edges)}")
new_edges = new_edges.drop_duplicates()
print(f"After dedup:      {len(new_edges)}")

print()
print("Edges per split")
split_lookup = dict(zip(new_pn['node_id'], new_pn['split']))
new_edges['split'] = new_edges['source'].map(split_lookup)
print(new_edges['split'].value_counts())
new_edges = new_edges.drop(columns=['split'])

print()
print(f"Year range: {new_edges['year'].min()} – {new_edges['year'].max()}")
print(f"Missing years: {new_edges['year'].isna().sum()}")

Raw edges:        635846
After dedup:      545311

Edges per split
split
SKG      263350
BLOG     224265
NOVEL     57696
Name: count, dtype: int64

Year range: 2022 – 2025
Missing years: 0


In [8]:
# Add new entities to entity_nodes
if new_entities:
    new_entity_rows = pd.DataFrame([
        {'node_id': eid, 'node_type': 'Entity', 'name': name}
        for name, eid in new_entities.items()
    ])
    print(f"Adding {len(new_entity_rows)} new entities to entity_nodes.csv")

    updated_entities = pd.concat([entity_nodes, new_entity_rows], ignore_index=True)
    updated_entities = updated_entities.drop_duplicates(subset='node_id')
    updated_entities.to_csv(OUT_DIR + 'entity_nodes.csv', index=False)
    print(f"entity_nodes.csv updated: {len(updated_entities)} total entities")
else:
    print("No new entities — entity_nodes.csv unchanged.")

Adding 151421 new entities to entity_nodes.csv
entity_nodes.csv updated: 293149 total entities


In [9]:
# Save
new_edges.to_csv(OUT_DIR + 'knowledge_edges_2022_2025.csv', index=False)
print(f"Saved: knowledge_edges_2022_2025.csv  ({len(new_edges)} edges)")
print(f"Papers with no edges: {len(set(new_pn['node_id']) - set(new_edges['source']))}")

Saved: knowledge_edges_2022_2025.csv  (545311 edges)
Papers with no edges: 1458
